# 04 · RGB single-crop fitting diagnostic

Fit a fresh RGB U-Net to the center crop of training image `India_285297`. This isolates fitting ability from changing crops and batches. The completed cloud diagnostic reached water IoU **0.3112 at step 200** and **0.9123 at step 1000**, on this training crop only.

The executable sequence retains the original 200-step phase and 800-step continuation, with one final visualization.

## 1. Fixed sample and configuration

| Setting | Value |
|---|---|
| Input / target | RGB [1,3,256,256]; labels [1,256,256] |
| Crop | India_285297, rows and columns 128:384 |
| Initialization | Seed 42; shared reference initialization from the baseline |
| Optimizer | Adam, learning rate 1e-4, weight decay 0 |
| Loss | Unweighted cross-entropy, ignore_index=-1 |
| Updates | 200, then continue the same model and optimizer to 1000 |
| Augmentation / precision | None / FP32 |

Only training data are used. Outputs are saved under a new `outputs/rgb_single_sample/` directory.

In [ ]:
from pathlib import Path
from datetime import datetime
import csv
import json
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
import torch
from torch import nn
from torch.nn import functional as F

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
BASE = ROOT / "data/sen1floods11/v1.1"
HAND = BASE / "data/flood_events/HandLabeled"
CFG = dict(seed=42, steps=200, log_every=20, lr=1e-4, weight_decay=0.0,
           sample="India_285297", crop_size=256, batch_size=1, amp=False)
device = torch.device("cuda")
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
OUT = ROOT / "outputs/rgb_single_sample" / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
OUT.mkdir(parents=True)
(OUT / "config.json").write_text(json.dumps({**CFG, "torch": str(torch.__version__)}, indent=2), encoding="utf-8")
with (BASE / "splits/flood_handlabeled/flood_train_data.csv").open(newline="") as f:
    rows = list(csv.reader(f))
image_name, label_name = next(row for row in rows if row[0] == CFG["sample"] + "_S1Hand.tif")
with rasterio.open(HAND / "S2Hand" / image_name.replace("_S1Hand", "_S2Hand")) as src:
    bands = [src.descriptions.index(b) + 1 for b in ["B4", "B3", "B2", "B8"]]
    x4 = src.read(bands, masked=True).astype("float32").filled(np.nan)
with rasterio.open(HAND / "LabelHand" / label_name) as src:
    label = src.read(1).astype("int64")
label[~np.isfinite(x4).all(axis=0)] = -1
x4 = np.nan_to_num(x4 / 10000.0)
size = CFG["crop_size"]
top, left = (label.shape[0] - size) // 2, (label.shape[1] - size) // 2
rgb = x4[:3, top:top+size, left:left+size].copy()
label = label[top:top+size, left:left+size].copy()
x = torch.from_numpy(rgb).unsqueeze(0).to(device)
y = torch.from_numpy(label).unsqueeze(0).to(device)
valid = y != -1
print("GPU:", torch.cuda.get_device_name(0))
print("Input:", tuple(x.shape), "valid pixels:", int(valid.sum()), "water pixels:", int((y == 1).sum()))
print("Output:", OUT)

## 2. Model and initialization

Use the original four-level U-Net with a 1024-channel bottleneck. Initialize from the same four-channel reference as Notebook 02, taking its first three input channels for RGB.

In [ ]:
import torch
from torch import nn

class DoubleConv(nn.Sequential):
    def __init__(self, in_channels, out_channels):
        super().__init__(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.ReLU(inplace=True),
        )

class UNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=2):
        super().__init__()
        self.encoders = nn.ModuleList([
            DoubleConv(in_channels, 64), DoubleConv(64, 128),
            DoubleConv(128, 256), DoubleConv(256, 512),
        ])
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(512, 1024)
        self.upsamples = nn.ModuleList([
            nn.ConvTranspose2d(1024, 512, 2, stride=2),
            nn.ConvTranspose2d(512, 256, 2, stride=2),
            nn.ConvTranspose2d(256, 128, 2, stride=2),
            nn.ConvTranspose2d(128, 64, 2, stride=2),
        ])
        self.decoders = nn.ModuleList([
            DoubleConv(1024, 512), DoubleConv(512, 256),
            DoubleConv(256, 128), DoubleConv(128, 64),
        ])
        self.head = nn.Conv2d(64, num_classes, 1)

    def forward(self, x):
        skips = []
        for encoder in self.encoders:
            x = encoder(x)
            skips.append(x)
            x = self.pool(x)
        x = self.bottleneck(x)
        for upsample, decoder, skip in zip(self.upsamples, self.decoders, reversed(skips)):
            x = upsample(x)
            x = decoder(torch.cat([skip, x], dim=1))
        return self.head(x)



torch.manual_seed(CFG["seed"])
reference = UNet(in_channels=4, num_classes=2)
state = {key: value.detach().clone() for key, value in reference.state_dict().items()}
del reference
state["encoders.0.0.weight"] = state["encoders.0.0.weight"][:, :3].clone()
model = UNet(in_channels=3, num_classes=2)
model.load_state_dict(state)
initial_first_layer = state["encoders.0.0.weight"].clone()
del state
model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])

## 3. First 200 updates

Update the fresh model on the same crop and record metrics every 20 steps. Save the step-200 weights, CSV and first-layer weight change. No validation selection is involved.

In [ ]:
def measure(step):
    model.eval()
    with torch.inference_mode():
        logits = model(x)
        loss = F.cross_entropy(logits, y, ignore_index=-1)
        p = logits.softmax(1)[:, 1]
        pred = logits.argmax(1)
        water = (y == 1)
        predicted = (pred == 1) & valid
        intersection = (predicted & water).sum()
        union = (predicted | water).sum()
        return dict(step=step, loss=float(loss), water_iou=float(intersection / union),
                    predicted_water_fraction=float(predicted.sum() / valid.sum()),
                    probability_on_water=float(p[water].mean()),
                    probability_on_nonwater=float(p[y == 0].mean()))

history = [measure(0)]
print(history[0], flush=True)
for step in range(1, CFG["steps"] + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    logits = model(x)
    loss = F.cross_entropy(logits, y, ignore_index=-1)
    loss.backward()
    optimizer.step()
    if step % CFG["log_every"] == 0 or step == CFG["steps"]:
        row = measure(step)
        history.append(row)
        with (OUT / "history.csv").open("w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=list(history[0]))
            writer.writeheader()
            writer.writerows(history)
        print(row, flush=True)

torch.save({"model": model.state_dict(), "config": CFG, "bands": ["B4", "B3", "B2"],
            "metrics": history[-1]}, OUT / "last.pt")
change = (model.encoders[0][0].weight.detach().cpu() - initial_first_layer).norm().item()
print("First-layer weight change (L2):", change)
print("Saved:", OUT)

## 4. Continue to 1000 updates

Continue with the live model and Adam state for another 800 updates, recording every 100 steps. Run this notebook sequentially: the step-200 checkpoint alone does not store the optimizer state needed for exact continuation after a kernel restart.

The final panels compare the fitted crop with its label and show the complete loss and IoU histories. These measure fitting, not test performance.

In [ ]:
TARGET_STEPS = 1000
start_step = history[-1]["step"]
continuation_dir = OUT / f"continue_{start_step}_to_{TARGET_STEPS}"
continuation_dir.mkdir(exist_ok=True)

for step in range(start_step + 1, TARGET_STEPS + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    loss = F.cross_entropy(model(x), y, ignore_index=-1)
    loss.backward()
    optimizer.step()
    if step % 100 == 0 or step == TARGET_STEPS:
        row = measure(step)
        history.append(row)
        with (continuation_dir / "history.csv").open("w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=list(history[0]))
            writer.writeheader()
            writer.writerows(history)
        print(row, flush=True)

continued_config = {**CFG, "steps": TARGET_STEPS, "continued_from_step": start_step}
torch.save({"model": model.state_dict(), "config": continued_config,
            "bands": ["B4", "B3", "B2"], "metrics": history[-1]}, continuation_dir / "last.pt")
OUT = continuation_dir
print("Saved continuation:", OUT)


In [ ]:
model.eval()
with torch.inference_mode():
    logits = model(x)
    probability = logits.softmax(1)[0, 1].cpu().numpy()
    prediction = logits.argmax(1)[0].cpu().numpy().astype("int16")
prediction[label == -1] = -1
probability[label == -1] = np.nan
steps = [row["step"] for row in history]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), layout="constrained")
axes[0].plot(steps, [row["loss"] for row in history])
axes[0].set(xlabel="Update step", ylabel="Cross entropy", title="Single training crop · loss")
axes[1].plot(steps, [row["water_iou"] for row in history])
axes[1].set(xlabel="Update step", ylabel="Water IoU", ylim=(0, 1), title="Single training crop · IoU")
fig.savefig(OUT / "fit_curve.png", dpi=150)
plt.show()
plt.close(fig)

rgb_display = np.moveaxis(rgb, 0, -1)
lo, hi = np.percentile(rgb_display[label != -1], [2, 98], axis=0)
rgb_display = np.clip((rgb_display - lo) / np.maximum(hi - lo, 1e-6), 0, 1)
cmap = ListedColormap(["#d7dee7", "#eee8d5", "#167caf"])
norm = BoundaryNorm([-1.5, -0.5, 0.5, 1.5], 3)
fig, axes = plt.subplots(1, 4, figsize=(13, 3.5), layout="constrained")
axes[0].imshow(rgb_display)
axes[1].imshow(label, cmap=cmap, norm=norm, interpolation="nearest")
axes[2].imshow(prediction, cmap=cmap, norm=norm, interpolation="nearest")
im = axes[3].imshow(probability, cmap="Blues", vmin=0, vmax=1)
for ax, title in zip(axes, ["RGB training crop", "Label", "RGB prediction", "Water probability"]):
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
fig.colorbar(im, ax=axes[3], shrink=0.8)
fig.savefig(OUT / "fit_prediction.png", dpi=150)
plt.show()
plt.close(fig)
print("Final training-crop metrics:", history[-1])